In [3]:
import torch
from yolov5.models.yolo import Model
from pathlib import Path
import os

In [57]:
ROOT_DIR = os.path.abspath(os.path.join(os.getcwd(), '..', '..'))
model_path = os.path.join(ROOT_DIR, 'models', 'plate_detector.pt')
in_path = os.path.join(ROOT_DIR, 'data', 'input 17k')
out_path = os.path.join(ROOT_DIR, 'data', 'output 17k')

In [5]:
device = torch.device('cuda')
model = torch.hub.load('ultralytics/yolov5', 'custom', path=model_path, force_reload=True)
model = model.to(device)

Downloading: "https://github.com/ultralytics/yolov5/zipball/master" to C:\Users\ADMIN/.cache\torch\hub\master.zip
YOLOv5  2025-4-18 Python-3.12.7 torch-2.6.0+cu124 CUDA:0 (NVIDIA GeForce RTX 3060, 12288MiB)

Fusing layers... 
Model summary: 213 layers, 7012822 parameters, 0 gradients, 15.8 GFLOPs
Adding AutoShape... 


In [6]:
import matplotlib.pyplot as plt
%matplotlib inline
import cv2
import re

In [59]:
model.conf = 0.70
padding = 20
min_px = 96

In [61]:
files = os.listdir(in_path)
existing_files = [file for file in os.listdir(out_path) if file.endswith('.jpg')]
indices = []

for file in existing_files:
    match = re.match(r"(\d+)\.jpg", file)
    if match:
        indices.append(int(match.group(1)))

idx = max(indices) + 1 if indices else 1

In [ ]:
"""
    Get the bounding box of the license plate from the image.
    Crop the license plate and save in 'normal' folder
"""

for image_file in files:
    img_path = os.path.join(in_path, image_file)
    img = cv2.imread(img_path)
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    result = model(img_rgb)

    boxes = result.xyxy[0]

    for box in boxes:
        x1, y1, x2, y2 = box[:4].cpu().numpy()
        x1 = max(0, int(x1) - padding)
        y1 = max(0, int(y1) - padding)
        x2 = min(img.shape[1], int(x2) + padding)
        y2 = min(img.shape[0], int(y2) + padding)

        # Make width and height divisible by 4
        width = (x2 - x1) - ((x2 - x1) % 4)
        height = (y2 - y1) - ((y2 - y1) % 4)
        x2 = x1 + width
        y2 = y1 + height

        # Ensure we don’t exceed image bounds
        x2 = min(x2, img.shape[1])
        y2 = min(y2, img.shape[0])

        # Draw padded bounding box
        cv2.rectangle(img_rgb, (x1, y1), (x2, y2), (255, 0, 0), 10)
        cropped_img = img[y1:y2, x1:x2]

        # Save cropped image if above 120 px
        if cropped_img.shape[0] > min_px and cropped_img.shape[1] > min_px:
            crop_filename = f'{out_path}/{idx}.jpg'
            cv2.imwrite(crop_filename, cropped_img)
            print(f'Saved cropped plate: {crop_filename} {image_file}')
            idx += 1

        else:
            print('failed', image_file)

        #plt.figure(figsize=(5, 5))
        #plt.imshow(img_rgb)
        #plt.axis('off')
        #plt.show();